# 1x1-conv-channel-reshape — ex1: prove 1x1 Conv2d equals per-pixel Linear

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `1x1-conv-channel-reshape`. Running the final beacon cell reports progress against the `CNN: 1x1 conv channel-reshape` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 1x1 conv channel-reshape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`1x1-conv-channel-reshape`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "1x1-conv-channel-reshape"
DD_SUBTOPIC = "CNN: 1x1 conv channel-reshape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 1×1 Conv as per-pixel Linear — quick refresher

`nn.Conv2d(IC, OC, kernel_size=1)` is exactly a **per-pixel linear** map from `IC` channels to `OC` channels:

```
y[b, oc, h, w] = sum_ic weight[oc, ic, 0, 0] * x[b, ic, h, w] + bias[oc]
```

There's no spatial mixing — each `(h, w)` position is processed independently. The op is identical to applying `nn.Linear(IC, OC)` after rearranging `(B, IC, H, W) → (B*H*W, IC)`, then rearranging back.

**Three uses in ResNet-family models:**

1. **Skip-branch projection** — match channel/stride between input and output of a residual block when they differ.
2. **Bottleneck blocks** (ResNet-50+) — squeeze with `1×1, IC→IC/4`, do the expensive `3×3, IC/4→IC/4`, then expand with `1×1, IC/4→IC`. Three convs but cheaper than one big 3×3.
3. **Channel reduction at the head** — collapse 2048 → 1000 with a single 1×1 conv instead of a flatten + Linear.

**Shape preservation.** `kernel_size=1` with `padding=0` and `stride=1` preserves H and W exactly: `(B, IC, H, W) → (B, OC, H, W)`. With `stride=s`, it downsamples by `s` while reshaping channels — that's the form ResNet's skip branch uses when the conv branch has `stride=s`.

**Parameter count vs full Linear.** `Conv2d(IC, OC, 1)` has `IC*OC + OC` params — exactly the same as `Linear(IC, OC)`. The 'conv' wrapping is just so the op accepts `(B, C, H, W)` directly without manual reshapes.

### Exercise 1 — prove 1x1 Conv2d equals per-pixel Linear

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the equivalence between `nn.Conv2d(IC, OC, kernel_size=1)` and `nn.Linear(IC, OC)` applied per-pixel by constructing matching weights and comparing outputs.
> Keywords: 1x1-conv, per-pixel-linear, channel-projection, resnet-bottleneck
> ```

**KCs targeted:** `1x1-conv-equals-per-pixel-linear`, `1x1-conv-shape-preservation`

Implement `ex1_one_by_one_via_linear(x, conv)`. Given:

- `x`: input tensor of shape `(B, IC, H, W)`.
- `conv`: an `nn.Conv2d(IC, OC, kernel_size=1)` module (already constructed by the test harness).

Return the result of applying a per-pixel `nn.Linear(IC, OC)` with the SAME weights as `conv` and producing a tensor of shape `(B, OC, H, W)`. Algorithm:

1. Build `linear = nn.Linear(IC, OC)`.
2. **Copy the conv weights into the linear's weight.** The conv's `weight` has shape `(OC, IC, 1, 1)`; `nn.Linear.weight` has shape `(OC, IC)`. The map is:
   ```
   linear.weight.data = conv.weight.data.view(OC, IC)
   linear.bias.data   = conv.bias.data
   ```
   Use `.data` so the assignment doesn't track gradients (we're just copying initial weights for comparison).

3. Rearrange `x` so each pixel is a row of features:
   ```
   x_flat = einops.rearrange(x, 'b c h w -> (b h w) c')   # (B*H*W, IC)
   ```

4. Apply the linear: `y_flat = linear(x_flat)`  → `(B*H*W, OC)`.

5. Rearrange back to image shape:
   ```
   y = einops.rearrange(y_flat, '(b h w) c -> b c h w', b=B, h=H, w=W)
   ```

Return `y`. The test compares your output to `conv(x)` to fp tolerance — they must be identical (it's literally the same computation, just written differently).

**Why this matters.** The two interpretations of 1×1 conv are equivalent — there's no special spatial trick. That's the atomic insight the drill is locking in.

In [ ]:
def ex1_one_by_one_via_linear(x: Tensor, conv) -> Tensor:
    import torch.nn as nn
    OC, IC, _, _ = conv.weight.shape
    B, _, H, W = x.shape
    linear = nn.Linear(IC, OC)
    linear.weight.data = conv.weight.data.view(OC, IC).clone()
    linear.bias.data   = conv.bias.data.clone()
    x_flat = einops.rearrange(x, 'b c h w -> (b h w) c')
    y_flat = linear(x_flat)
    return einops.rearrange(y_flat, '(b h w) c -> b c h w', b=B, h=H, w=W)


<details><summary>Solution</summary>

```python
def ex1_one_by_one_via_linear(x: Tensor, conv) -> Tensor:
    import torch.nn as nn
    OC, IC, _, _ = conv.weight.shape
    B, _, H, W = x.shape
    linear = nn.Linear(IC, OC)
    linear.weight.data = conv.weight.data.view(OC, IC).clone()
    linear.bias.data   = conv.bias.data.clone()
    x_flat = einops.rearrange(x, 'b c h w -> (b h w) c')
    y_flat = linear(x_flat)
    return einops.rearrange(y_flat, '(b h w) c -> b c h w', b=B, h=H, w=W)
```

**Why the shapes line up.** Conv2d weight is `(OC, IC, K, K)`. For `K=1`, that's `(OC, IC, 1, 1)` — the two trailing size-1 axes hold zero information. `view(OC, IC)` collapses them. `nn.Linear.weight` is `(out_features, in_features)` = `(OC, IC)` — exact match. The conv's bias is already shape `(OC,)` matching Linear's.

**Why `.clone()`.** Using `.data = ...` shares storage by default (when shapes match). Cloning makes the test's assertion 'modifying linear doesn't affect conv' true if you ever extend the drill — it's a habit-forming defensive copy. Not strictly required for the test as written.

**Per-element formula identity.** For both ops:
```
y[b, oc, h, w] = sum_ic W[oc, ic] * x[b, ic, h, w] + bias[oc]
```
1×1 Conv2d and per-pixel Linear are LITERALLY the same computation. PyTorch's CUDA backends just happen to dispatch them through different kernels for memory-layout reasons.

**Why this matters for ResNet bottlenecks.** The bottleneck block `Conv(1x1, IC→IC/4) → Conv(3x3, IC/4→IC/4) → Conv(1x1, IC/4→IC)` is 'squeeze, mix spatially, expand' — but you can also read it as 'per-pixel Linear projection, conv, per-pixel Linear projection'. That mental model is what makes transformer FFN blocks (`Linear → activation → Linear`) and CNN bottlenecks feel like the same primitive.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()